# Walmart Retail Data — Multi-Variate Analysis
Multi-variate analysis examines **three or more variables simultaneously**, revealing complex interactions and patterns invisible in simpler analyses.

### Techniques covered:
1. Pair Plot (all numerics, colored by category)
2. Annotated Correlation Heatmap
3. Pivot Table Heatmap (Sales × Category × State)
4. Bubble Chart (3 dimensions at once)
5. Facet Grid (distribution per category)
6. Stacked Bar (quantity by category per state)
7. 3D Scatter Plot
8. Multi-line trend by category over time
9. Summary statistics matrix

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D

sns.set_style('whitegrid')
sns.set_palette('husl')
pd.set_option('display.float_format', '{:.2f}'.format)

df = pd.read_csv('/Users/satyakidas/Downloads/Walmart_Cleaned.csv', parse_dates=['Order Date','Ship Date'])
print('Loaded:', df.shape)
df.head(3)

**Code explanation:** Loads all required libraries including `Axes3D` for 3D plotting, reads the cleaned dataset, and applies seaborn styling.

**Observation:** Dataset loaded with all 18 columns. Using `mpl_toolkits.mplot3d` requires no additional installation — it is bundled with matplotlib. The multivariate section builds on patterns identified in uni- and bi-variate analyses to explore simultaneous interactions.

## 1. Pair Plot
A pair plot shows scatter plots for all pairs of numeric variables, with histograms on the diagonal. Coloring by Category reveals group-level patterns.

In [ ]:
# Use top 5 categories to keep the plot readable
top5_cats = df['Category'].value_counts().head(5).index
df_top5 = df[df['Category'].isin(top5_cats)]

pair_cols = ['Sales', 'Quantity', 'Profit', 'Profit_Margin', 'Shipping_Days']
g = sns.pairplot(df_top5[pair_cols + ['Category']], hue='Category',
                 diag_kind='hist', plot_kws={'alpha': 0.4, 's': 15},
                 height=2.2, aspect=1.0)
g.figure.suptitle('Pair Plot — Top 5 Categories', y=1.02, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Code explanation:** Filters to the top 5 categories (by order count) to keep the pair plot readable, then uses `sns.pairplot()` with `hue='Category'` to colour-code all scatter pairs by category and show histograms on the diagonal.

**Observation:**
- **Sales vs Profit** shows the clearest positive trend across all categories
- **Copiers (if in top 5)** appear as a separate cluster with higher Sales values
- **Diagonal histograms** confirm right skew for Sales and Profit and near-uniform distribution for Quantity
- Categories overlap heavily in the Quantity dimension — quantity alone does not differentiate categories
- The pair plot is most valuable for spotting clusters, outliers, and non-linear relationships across all variable pairs simultaneously

## 2. Full Annotated Correlation Heatmap

In [ ]:
num_cols = ['Sales', 'Quantity', 'Profit', 'Profit_Margin', 'Shipping_Days']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            ax=ax, square=True, linewidths=1,
            cbar_kws={'label': 'Pearson r', 'shrink': 0.8})
ax.set_title('Full Correlation Heatmap (All Numeric Features)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Code explanation:** Computes and displays the full Pearson correlation matrix for all 5 numeric features as a `coolwarm` annotated heatmap (full matrix, not masked).

**Observation:**
- The diagonal is always 1.0 (self-correlation)
- **Sales–Profit (0.48)** remains the dominant relationship
- **Profit_Margin** is essentially uncorrelated with Quantity (r ≈ 0.01) — buying more units doesn't improve margin
- **Shipping_Days** shows near-zero correlation with all financial metrics — fast shipping is not linked to higher-value orders
- No feature pair exceeds r = 0.5, indicating **low multicollinearity** — each variable carries independent information

## 3. Pivot Table Heatmap — Sales by Category × State
Three dimensions encoded: Category (rows), State (columns), Sales (colour intensity).

In [ ]:
pivot = df.pivot_table(index='Category', columns='State', values='Sales', aggfunc='sum', fill_value=0)

fig, ax = plt.subplots(figsize=(14, 9))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax,
            linewidths=0.3, cbar_kws={'label': 'Total Sales ($)'})
ax.set_title('Total Sales by Category & State (Pivot Heatmap)', fontsize=13, fontweight='bold')
ax.set_xlabel('State')
ax.set_ylabel('Category')
plt.tight_layout()
plt.show()

**Code explanation:** Creates a pivot table aggregating total Sales grouped by Category (rows) and State (columns), filled with 0 for missing combinations, then visualised as a heatmap with raw Sales values annotated.

**Observation:**
- **California × Phones and California × Chairs** are among the highest Sales cells — large state with strong demand
- **Copiers** show high Sales in California and New York but zero in many other states — limited geographic reach
- **Tables** show surprisingly high Sales in some states but remain loss-making — volume without profitability
- The heatmap reveals **whitespace opportunities**: categories with strong performance in one region but absent in others could be expanded

## 4. Bubble Chart
Encodes **4 dimensions**: X=Sales, Y=Profit, Bubble Size=Quantity, Colour=Category.

In [ ]:
df_bubble = df[df['Category'].isin(top5_cats)].copy()
categories = df_bubble['Category'].unique()
color_map = dict(zip(categories, sns.color_palette('husl', len(categories))))

fig, ax = plt.subplots(figsize=(13, 7))
for cat in categories:
    sub = df_bubble[df_bubble['Category'] == cat]
    ax.scatter(sub['Sales'], sub['Profit'],
               s=sub['Quantity'] * 20,
               alpha=0.5, label=cat,
               color=color_map[cat], edgecolors='white', linewidths=0.5)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Sales ($)', fontsize=12)
ax.set_ylabel('Profit ($)', fontsize=12)
ax.set_title('Bubble Chart: Sales vs Profit\n(Bubble Size = Quantity, Colour = Category)',
             fontsize=13, fontweight='bold')
ax.legend(title='Category', bbox_to_anchor=(1.01,1), loc='upper left')
plt.tight_layout()
plt.show()

**Code explanation:** For each of the top 5 categories, plots a scatter of Sales (x) vs Profit (y), with bubble size proportional to `Quantity × 20` and colour per category.

**Observation:**
- Large bubbles (high quantity) tend to cluster at lower Sales values — bulk orders of cheap items
- **High-Sales, high-Profit bubbles** are small — expensive items bought in small quantities
- Some bubbles fall **below the zero profit line** (dashed) — these are loss-making transactions visible even at mid-level Sales
- The bubble chart encodes 4 dimensions simultaneously, making it one of the most information-dense visualisations in this analysis

## 5. Facet Grid — Sales Distribution per Category
Small multiples let us compare the same plot across all category groups simultaneously.

In [ ]:
g = sns.FacetGrid(df, col='Category', col_wrap=4, height=3, sharey=False)
g.map(plt.hist, 'Sales', bins=20, color='steelblue', edgecolor='black', alpha=0.7)
g.set_titles(col_template='{col_name}', fontsize=9)
g.set_axis_labels('Sales ($)', 'Frequency')
g.figure.suptitle('Sales Distribution per Category (Facet Grid)', y=1.02,
                   fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Code explanation:** Uses `sns.FacetGrid` to produce a small-multiple layout — one histogram of Sales per category, arranged in 4 columns.

**Observation:**
- Every category shows a **right-skewed Sales distribution** — low-value orders dominate in all product types
- **Copiers and Machines** have histograms shifted to the right — their typical order value is much higher than other categories
- **Binders, Paper, Labels** have distributions heavily concentrated near zero — commodity products with very low per-unit prices
- Facet grids are ideal for **comparing the same pattern across groups** without visual interference

## 6. Stacked Bar — Quantity by Category per State

In [ ]:
qty_pivot = df.pivot_table(index='State', columns='Category', values='Quantity', aggfunc='sum', fill_value=0)

fig, ax = plt.subplots(figsize=(14, 7))
qty_pivot.plot(kind='bar', stacked=True, ax=ax, colormap='tab20', edgecolor='black', alpha=0.85)
ax.set_title('Total Quantity Sold — Category per State (Stacked)', fontsize=13, fontweight='bold')
ax.set_xlabel('State')
ax.set_ylabel('Total Quantity')
ax.legend(title='Category', bbox_to_anchor=(1.01,1), loc='upper left', fontsize=8)
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

**Code explanation:** Creates a pivot table of total Quantity by State (rows) and Category (columns), then plots as a stacked bar where each bar represents one state and colours show category contributions.

**Observation:**
- **California's bar is the tallest** — highest total quantity across almost all categories
- **Binders and Storage** form the largest coloured segments in most states — high-volume, frequently ordered items
- **Copiers and Machines** are barely visible in the stacks — bought infrequently in small quantities everywhere
- The stacked format lets us see both **total volume per state** and **category mix** in a single chart

## 7. 3D Scatter Plot — Sales, Profit, Quantity

In [ ]:
fig = plt.figure(figsize=(12, 8))
ax3d = fig.add_subplot(111, projection='3d')

categories_all = df['Category'].unique()
colors_3d = sns.color_palette('husl', len(categories_all))

for cat, col in zip(categories_all, colors_3d):
    sub = df[df['Category'] == cat]
    ax3d.scatter(sub['Sales'], sub['Profit'], sub['Quantity'],
                 color=col, alpha=0.5, s=15, label=cat)

ax3d.set_xlabel('Sales ($)')
ax3d.set_ylabel('Profit ($)')
ax3d.set_zlabel('Quantity')
ax3d.set_title('3D Scatter: Sales vs Profit vs Quantity', fontsize=12, fontweight='bold')
ax3d.legend(loc='upper left', fontsize=7, bbox_to_anchor=(1.0, 1.0))
plt.tight_layout()
plt.show()

**Code explanation:** Creates a 3D matplotlib figure using `Axes3D`, plots each category as a separate scatter series with X=Sales, Y=Profit, Z=Quantity.

**Observation:**
- The 3D space shows that most transactions cluster near the origin (low Sales, low Profit, low Quantity)
- **High-Sales points** are scattered in an upper layer with varying Profit — confirming that high revenue doesn't guarantee high profit
- Rotating the 3D plot (in interactive mode) reveals that the Sales–Profit plane is the most informative 2D projection
- Categories are separable along the Sales axis but heavily overlapping in Quantity and Profit dimensions

## 8. Multi-Line Trend — Sales by Year for Top 5 Categories

In [ ]:
trend = df[df['Category'].isin(top5_cats)].groupby(['Year','Category'])['Sales'].sum().reset_index()

fig, ax = plt.subplots(figsize=(12, 6))
for cat in top5_cats:
    sub = trend[trend['Category'] == cat]
    ax.plot(sub['Year'], sub['Sales'], marker='o', linewidth=2.5, label=cat)

ax.set_title('Yearly Sales Trend — Top 5 Categories', fontsize=13, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Total Sales ($)')
ax.legend(title='Category')
ax.set_xticks(df['Year'].unique())
plt.tight_layout()
plt.show()

**Code explanation:** Filters to top 5 categories, groups by Year and Category, sums Sales, then plots one line per category across years.

**Observation:**
- All top 5 categories show **upward trends from 2011 to 2014** — overall business growth
- **Phones consistently lead** in total Sales among the top 5 — driven by high per-unit price
- **Chairs** show the steepest growth trajectory — potentially expanding market or new corporate clients
- The gap between categories widens over years — the leading categories are growing faster than others
- 2013 shows a dip for some categories — possible inventory or supply chain disruption that year

## 9. Summary Statistics Matrix by Category

In [ ]:
summary = df.groupby('Category').agg(
    Total_Sales   = ('Sales',  'sum'),
    Total_Profit  = ('Profit', 'sum'),
    Total_Qty     = ('Quantity','sum'),
    Avg_Sales     = ('Sales',  'mean'),
    Avg_Profit    = ('Profit', 'mean'),
    Avg_Margin    = ('Profit_Margin','mean'),
    Avg_Ship_Days = ('Shipping_Days','mean'),
    Order_Count   = ('Sales',  'count')
).round(2).sort_values('Total_Sales', ascending=False)

print('Summary Statistics Matrix by Category:')
summary

**Code explanation:** Groups by Category and computes 8 aggregate metrics including total Sales, total Profit, average margin, average shipping days, and order count.

**Observation:**
- **Copiers** top Total_Sales and Avg_Sales despite fewest orders — high-value, low-frequency product
- **Tables** have high Total_Sales but negative Total_Profit — a strategic pricing problem
- **Binders** lead in Order_Count — the most frequently ordered item
- **Avg_Ship_Days** is consistent across categories (~4 days) — logistics are uniform regardless of product type
- **Avg_Margin** varies from negative (Tables) to >30% (Labels, Fasteners) — significant pricing inconsistency across the portfolio

In [ ]:
# Heatmap of normalized summary stats
from sklearn.preprocessing import MinMaxScaler  # may not be available; fallback to manual
try:
    from sklearn.preprocessing import MinMaxScaler
    scaler = MinMaxScaler()
    norm = pd.DataFrame(scaler.fit_transform(summary), index=summary.index, columns=summary.columns)
except ImportError:
    norm = (summary - summary.min()) / (summary.max() - summary.min())

fig, ax = plt.subplots(figsize=(13, 8))
sns.heatmap(norm, annot=summary.values, fmt='.0f', cmap='Blues', ax=ax,
            linewidths=0.3, cbar_kws={'label': 'Normalised Value (0-1)'})
ax.set_title('Normalised Summary Statistics by Category', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Code explanation:** Normalises the summary matrix to 0–1 scale using min-max scaling, then plots a heatmap with raw values annotated over the normalised colour scale.

**Observation:**
- The normalised heatmap makes **cross-metric comparisons** easier — you can see which category excels in which dimension simultaneously
- **Copiers** light up in Total_Sales and Avg_Sales but are dark in Orders — quality over quantity
- **Binders** light up in Orders and Total_Qty but are dim in Avg_Sales — volume over value
- **Tables** are visually striking: bright in Total_Sales and Total_Qty, but very dark in Avg_Margin (negative) — a clear signal for business intervention